

# FDTD Discretization of Equations

For the discretization of the equations, we introduce a notation that describes the space-time localization of the fields. The index $q$ is not a power, but a time step.

$$
\begin{align*}
E_z(x,t) &= E_z(m \, \Delta x, q \, \Delta t) = E_z^q[m] \\
H_y(x,t) &= H_y(m \, \Delta x, q \, \Delta t) = H_y^q[m]
\end{align*}
$$

### Explanation of Symbols

* **$E_z(x,t), H_y(x,t)$**: The continuous electric and magnetic field components as functions of continuous space ($x$) and time ($t$).
* **$m$**: The integer index representing a discrete position on the spatial grid. The actual position is $x = m \cdot \Delta x$.
* **$q$**: The integer index representing a discrete step in time. The actual time is $t = q \cdot \Delta t$.
* **$\Delta x$**: The spatial step size, i.e., the physical distance between adjacent points on the grid.
* **$\Delta t$**: The time step size, i.e., the duration between consecutive moments at which the fields are calculated.
* **$E_z^q[m], H_y^q[m]$**: The discretized values of the electric and magnetic fields at spatial index $m$ and time index $q$. This is the notation used in the computer algorithm.




## Discretization of Faraday's Law

In the space-time coordinate $[(m+1/2)\Delta x, q \Delta t]$, we can write for Faraday's law:

$$
\mu \frac{\partial H_y}{\partial t} \bigg|_{(m+1/2)\Delta x, \, q \Delta t}  = 
\frac{\partial E_z}{\partial x} \bigg|_{(m+1/2)\Delta x, \, q \Delta t}
$$

The derivative of $H_y$ at time $q \Delta t$ can be approximated using two consecutive values:

$$\mu \frac{\partial H_y}{\partial t} \bigg|_{(m+1/2)\Delta x, \, q \Delta t}  \approx \mu \frac{H_y^{q+1/2}[m+1/2] - H_y^{q-1/2}[m+1/2]}{\Delta t}$$

The derivative of $E_z$ at position $(m+1/2)\Delta x$ can be approximated using two neighboring values:

$$\frac{\partial E_z}{\partial x} \bigg|_{(m+1/2)\Delta x, \, q \Delta t} \approx \frac{E_z^{q}[m+1] - E_z^{q}[m] }{\Delta x}$$

From the above, it then follows:

$$\mu \frac{H_y^{q+1/2}[m+1/2] - H_y^{q-1/2}[m+1/2]}{\Delta t} = \frac{E_z^{q}[m+1] - E_z^{q}[m] }{\Delta x}$$

After rearranging, we get the **update equation** for the value $H_y^{q+1/2}[m+1/2]$ in the form:

$$H_y^{q+1/2}[m+1/2] = H_y^{q-1/2}[m+1/2] + \frac{\Delta t}{\mu \, \Delta x} \left( E_z^{q}[m+1] - E_z^{q}[m] \right)$$

### Explanation of Symbols

* **$H_y^{q+1/2}[m+1/2]$**: The value of the magnetic field at the future time step ($q+1/2$), which we want to calculate.
* **$H_y^{q-1/2}[m+1/2]$**: The value of the magnetic field at the previous time step ($q-1/2$), which is already known.
* **$E_z^{q}[m+1], E_z^{q}[m]$**: The known values of the electric field at neighboring points of the spatial grid at the current time step ($q$).
* **$\Delta t$**: The time step, the duration of one simulation step.
* **$\Delta x$**: The spatial step, the distance between two adjacent grid points.
* **$\mu$**: The permeability of the medium.

This *update equation* is the foundation of the FDTD method. It allows us to calculate the future value of the magnetic field based on known values from the past, thereby advancing the simulation in time.



<img src="./img/fdtd_01.png" width=600px alt="Aktualizačná rovnica pre Faradayov zákon" scale="0.8"/>

---
The space-time distribution of the field components $E_z$ and $H_y$ can then be illustrated in the following diagram, where $\otimes$ marks the point $[(m+1/2)\Delta x, q \Delta t]$. It is clear that the future value of the magnetic field depends on its previous value and the neighboring values of the electric field.




## Discretization of Ampère's Law

Following the update of the magnetic field, the next step in the FDTD algorithm is to update the electric field. This is achieved by discretizing Ampère's Law. The approximation is centered at the space-time coordinate $[m \Delta x, (q+1/2) \Delta t]$, which is spatially co-located with an E-field node but temporally halfway between two time steps.

For Ampère's law, we can write:

$$
\epsilon \frac{\partial E_z}{\partial t} \bigg|_{m \Delta x, \, (q+1/2) \Delta t}  = 
\frac{\partial H_y}{\partial x} \bigg|_{m \Delta x, \, (q+1/2) \Delta t}
$$

<!-- MEDSKIP -->

Using central-difference approximations for the derivatives, the equation becomes:

$$
\epsilon \frac{E_z^{q+1}[m] - E_z^{q}[m]}{\Delta t} = \frac{H_y^{q + 1/2}[m+1/2] - H_y^{q+1/2}[m-1/2] }{\Delta x}
$$

<!-- MEDSKIP -->

After rearranging, we get the **update equation** for the value $E_z^{q+1}[m]$ in the form:

$$
E_z^{q+1}[m] = E_z^{q}[m] + \frac{\Delta t}{\epsilon \, \Delta x} \left({H_y^{q + 1/2}[m + 1/2] - H_y^{q+1/2}[m-1/2] } \right)
$$

<!-- MEDSKIP -->

### Explanation and the Leapfrog Update

This equation is the second half of the FDTD **leapfrog algorithm**. It shows that the future value of the electric field ($E_z^{q+1}$) at a grid point `m` is determined by:
1.  Its own previous value ($E_z^{q}[m]$).
2.  The spatial difference of the two neighboring magnetic field values ($H_y^{q+1/2}[m+1/2]$ and $H_y^{q+1/2}[m-1/2]$) which were calculated in the *previous half-step*.

This creates a leapfrog sequence:
1.  Use known **E-fields** at time `q` to calculate **H-fields** at time `q + 1/2`.
2.  Use these new **H-fields** at time `q + 1/2` to calculate **E-fields** at time `q + 1`.
3.  Repeat for the duration of the simulation.

### Explanation of Symbols

* **$E_z^{q+1}[m]$**: The future value of the electric field at spatial index `m` and time index `q+1`, which we want to calculate.
* **$E_z^{q}[m]$**: The known, previous value of the electric field at the same location.
* **$H_y^{q + 1/2}[m + 1/2], H_y^{q+1/2}[m-1/2]$**: The known values of the magnetic field at neighboring grid points, calculated at the intermediate half-time step.
* **$\Delta t$**: The time step size.
* **$\Delta x$**: The spatial step size.
* **$\epsilon$**: The permittivity of the medium.


Similar to the discretization of Faraday's law, the future value of the electric field depends on its previous value and the neighboring values of the magnetic field

<img src="./img/fdtd_02.png" width=600px alt="Aktualizačná rovnica pre Ampérov zákon" scale="0.8"/>




## The Courant Number

The coefficients in the update equations, $\Delta t / \mu \Delta x$ and $\Delta t / \epsilon \Delta x$, are fundamental to the FDTD algorithm. They determine how far electromagnetic energy can propagate through the grid in a single time step. The magnitude of these coefficients influences the stability of the numerical solution and also defines the dimensional parameters of the simulation model.

Using the following fundamental relationships:

$$
\begin{align*}
c &= \frac{1}{\sqrt{\epsilon_0 \mu_0}} && \text{(Speed of light in vacuum)} \\
z_0 &= \sqrt{\frac{\mu_0}{\epsilon_0}} && \text{(Impedance of free space)} \\
\mu &= \mu_0 \mu_r && \text{(Permeability of the medium)} \\
\epsilon &= \epsilon_0 \epsilon_r && \text{(Permittivity of the medium)}
\end{align*}
$$

we can rewrite the coefficients in a more insightful form.

### Derivation of the Coefficients

**For the Electric Field Update Equation:**

The coefficient can be expanded and simplified as follows:

$$
\frac{\Delta t}{\epsilon \Delta x} =
\frac{1}{\epsilon_0 \epsilon_r} \frac{ \sqrt{\epsilon_0 \mu_0}}{ \sqrt{\epsilon_0 \mu_0}} \frac{\Delta t}{\Delta x} =
\frac{\sqrt{\epsilon_0 \mu_0}}{\epsilon_0 \epsilon_r} \frac{c \Delta t}{\Delta x} =
\frac{1}{\epsilon_r} \sqrt{\frac{\mu_0}{\epsilon_0}} \frac{c \Delta t}{\Delta x} = \frac{z_0}{\epsilon_r} S_c
$$

**For the Magnetic Field Update Equation:**

Similarly, the coefficient for the magnetic field is rewritten as:

$$
\frac{\Delta t}{\mu \Delta x} =
\frac{1}{\mu_0 \mu_r} \frac{ \sqrt{\epsilon_0 \mu_0}}{ \sqrt{\epsilon_0 \mu_0}} \frac{\Delta t}{\Delta x} =
\frac{\sqrt{\epsilon_0 \mu_0}}{\mu_0 \mu_r} \frac{c \Delta t}{\Delta x} =
\frac{1}{\mu_r} \sqrt{\frac{\epsilon_0}{\mu_0}} \frac{c \Delta t}{\Delta x} =
\frac{1}{\mu_r \, z_0} S_c
$$

where $S_c$ is a quantity called the **Courant number**, defined as:

$$S_c = \frac{c \Delta t}{\Delta x}$$

### Significance of the Courant Number

The Courant number is a dimensionless quantity that is critical for the stability and accuracy of the FDTD simulation.

* **Physical Meaning**: It represents the ratio of the distance a physical wave travels in one time step ($c \Delta t$) to the size of a spatial grid cell ($\Delta x$).
* **Stability Condition**: For the simulation to remain numerically stable and not produce infinitely growing, non-physical results, the Courant number must be less than or equal to a certain limit. In a 1D simulation, this limit is 1 ($S_c \le 1$). This condition, known as the **Courant-Friedrichs-Lewy (CFL) condition**, ensures that the numerical algorithm can "keep up" with the physical wave it is simulating.
* **Model Scaling**: By expressing the update coefficients in terms of $S_c$, the simulation becomes independent of the absolute size of $\Delta x$ and $\Delta t$. One only needs to define their ratio, which allows the results of a single simulation to be scaled to any physical size.

### Explanation of Symbols
* **$c$**: The speed of light in a vacuum.
* **$z_0$**: The intrinsic impedance of free space (~377 Ω).
* **$\mu, \epsilon$**: The permeability and permittivity of the medium.
* **$\mu_r, \epsilon_r$**: The relative permeability and permittivity of the medium.
* **$\mu_0, \epsilon_0$**: The permeability and permittivity of free space.
* **$\Delta t, \Delta x$**: The time step and spatial step (grid cell size).
* **$S_c$**: The Courant number (dimensionless).



## Choosing the Value of the Courant Number ($S_c$)

In the FDTD method, the relationship between the size of the spatial step ($\Delta x$) and the time step ($\Delta t$) is crucial. This relationship, known as the **Courant stability condition** (or Courant-Friedrichs-Lewy, CFL condition), ensures that the numerical solution remains stable and physically meaningful.

### Physical Basis

The condition arises from a simple causal consideration: the numerical speed of information propagation in the grid must be able to "keep up" with the physical speed of wave propagation. In other words, in one time step $\Delta t$, an electromagnetic wave propagating at the speed of light $c$ must not travel more than one spatial cell $\Delta x$ in the numerical grid. If this were to happen, the algorithm would not be able to correctly calculate the influence of neighboring points, and the numerical solution would become unstable (values would grow exponentially to infinity).

### The Courant Condition in N-Dimensions

An electromagnetic wave in free space cannot propagate at a speed greater than the speed of light $c$.

* **In the one-dimensional case (1D):**
  The time required for a wave to travel the size of one cell requires a minimum time of $\Delta t = \Delta x / c$. Therefore, it must hold that:
  
  $$
  c \Delta t \le \Delta x \quad \implies \quad S_c = \frac{c \Delta t}{\Delta x} \le 1
  $$

* **In the two-dimensional case (2D):**
  If a wave propagates along the diagonal of a cell (with equal sides $\Delta x = \Delta y$), in time $\Delta t$ it travels a distance of $\sqrt{(\Delta x)^2 + (\Delta y)^2} = \sqrt{2}\Delta x$. The stability condition is then tightened:
  
  $$
  c \Delta t \le \frac{\Delta x}{\sqrt{2}} \quad \implies \quad S_c = \frac{c \Delta t}{\Delta x} \le \frac{1}{\sqrt{2}} \approx 0.707
  $$

* **In the three-dimensional case (3D):**
  Similarly, for propagation along the space diagonal of a cube ($\Delta x = \Delta y = \Delta z$), it holds that:
  
  $$
  c \Delta t \le \frac{\Delta x}{\sqrt{3}} \quad \implies \quad S_c = \frac{c \Delta t}{\Delta x} \le \frac{1}{\sqrt{3}} \approx 0.577
  $$

By generalizing for an $n$-dimensional space with equal steps, we get the **Courant condition**:

$$
c \Delta t \leq \frac{\Delta x}{\sqrt{n}}
$$

### Practical Consequences and the Choice of $\Delta t$

This condition practically means that the time step $\Delta t$ must be sufficiently small compared to the spatial step $\Delta x$.

* **Stability vs. Computation Time**: A smaller value of $\Delta t$ means that a larger number of steps is required to simulate a given time interval. This leads to finer time resolution and greater solution stability, but at the cost of longer computation time.

* **Optimal Choice**: It is usually not necessary to choose $\Delta t$ much smaller than the stability limit. It is optimal to choose the time step close to the maximum allowed value (e.g., 95% to 99% of the limit) to maximize computational efficiency.

* **Normalized Units**: In the FDTD method, for practical reasons, one often works in a normalized system where the value of the speed of light $c$ is set to 1 ($c=1$). In this case, the stability condition simplifies to $\Delta t \le \Delta x / \sqrt{n}$. As a starting value in a 1D simulation, we then usually choose $S_c = 1$, which is called the "magic time step" because it minimizes numerical dispersion. For 2D and 3D simulations, however, $S_c$ must necessarily be less than 1.

**Všeobecný postup** výpočtu elektromagnetických polí je možné defnovať nasledujúcimi krokmi

- vo zvolenom priestore definujeme vybrané uzly, v ktorých budeme určovať hodnoty polí $E$ a $H$ 
- aproximujeme časové a priestorové derivácie v Ampérovom a Faradayovom zákone konečnými diferenciami 
- vyriešime sústavu diferenčných rovníc tak, aby sme získali *aktualizačné* rovnice, ktoré popisujú budúce hodnoty na základe predchádzajúcich hodnôt v priestore a čase
- v jednotlivých časových krokoch spočítame hodnoty polí $E$,$H$ pre všetky uzly priestoru
- opakujeme predchádzajúci krok, kým nedosiahneme stanovený čas ukončenia simulácie

-

## General Procedure of the FDTD Method

The calculation of electromagnetic fields using the FDTD method can be summarized in the following fundamental steps, which form the basis of every FDTD algorithm.

---

#### 1. Definition of the Computational Domain and Grid
* **Step:** In a chosen space, we define selected nodes where we will determine the values of the **E** and **H** fields.
* **Explanation:** The first step is the **discretization** of the problem. The continuous space in which the wave propagates is replaced by a finite three-dimensional grid of cells (often called Yee cells). The values of the electric and magnetic fields are not calculated everywhere, but only at specific points (nodes) on this grid. The components of the E-field and H-field are spatially offset from each other by half a spatial step, which is a key feature of the Yee algorithm.

---

#### 2. Approximation of Maxwell's Equations
* **Step:** We approximate the time and space derivatives in Ampère's and Faraday's laws with finite differences.
* **Explanation:** The continuous partial derivatives (e.g., $\partial/\partial t$, $\partial/\partial x$) found in Maxwell's equations are replaced by their algebraic approximations. The **central-difference** method is most commonly used because it is simple and provides second-order accuracy. This means that the continuous differential equations are transformed into a system of simple algebraic equations.

---

#### 3. Derivation of the Update Equations
* **Step:** We solve the system of difference equations to obtain **update equations**, which describe future values based on previous values in space and time.
* **Explanation:** After manipulating the algebraic equations from the previous step, we obtain explicit formulas. These *update equations* allow us to directly calculate the value of a field at a specific node at a future moment in time ($t + \Delta t$) based on its own previous value and the values of neighboring fields at the current and previous time steps. This process is often called a "leapfrog" algorithm because the calculation of E-fields and H-fields alternates in time.

---

#### 4. Computational Loop in the Time Domain
* **Step:** In individual time steps, we calculate the values of the **E** and **H** fields for all nodes in the space.
* **Explanation:** This is the core of the simulation. The program runs in a time loop. In each iteration (time step), the update equations are successively applied to all nodes in the entire computational grid. First, all H-field values for time step $q+1/2$ are calculated based on the known E-fields from step $q$. Subsequently, these new H-field values are used to calculate all E-fields for time step $q+1$.

---

#### 5. Termination of the Simulation
* **Step:** We repeat the previous step until we reach the specified end time of the simulation.
* **Explanation:** The time loop is repeated a predefined number of times (e.g., 1000 time steps). This number must be large enough to allow us to observe the entire course of the electromagnetic phenomenon of interest—for example, the propagation of a pulse through the entire domain, its reflection from an obstacle, and the settling of the field.